# Agentic financial research

Multi-agent workflow for issuer research. You ask a question in plain language; the system resolves the company, gathers market facts through tools, and returns a structured research brief.

**Pipeline**
1. **Research agent (A)** — chooses which data tools to call (price, volatility, news, sentiment, web search) and writes a typed `DataBrief`.
2. **Critic agent (B)** — reviews the brief, may request one missing fact, then publishes a `FinalReport`.
3. **Memory** — follow-up questions on the same issuer reuse stored facts; tools run only when new data is needed.

**Example questions:** `what is the news for apple`, `latest AAPL price`, or a full research request such as analysing financial health, market sentiment, 90-day risks, and a vol-grounded hedge for a ticker.

**API keys:** set `OPENROUTER_API_KEY` (preferred) and/or `GROQ_API_KEY` in Colab Secrets or a local `.env` file. Do not paste keys into notebook cells.

In [6]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Submission monorepo — clone once, then enter the Task 3 subfolder.
REPO_URL = os.environ.get(
    "SUBMISSION_REPO",
    "https://github.com/lavanblavan/-CDAZZDEV-MLE-Lavan.git",
)
TASK_SUBDIR = "task3_agentic"

def ensure_project() -> Path:
    if IN_COLAB:
        repo_root = Path("/content/CDAZZDEV-MLE-Lavan")
        root = repo_root / TASK_SUBDIR
        if not (root / "src/config.py").exists():
            if not repo_root.exists():
                subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
            if not (root / "src/config.py").exists():
                raise FileNotFoundError(
                    f"Expected {TASK_SUBDIR}/ under cloned repo at {repo_root}"
                )
        else:
            subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], check=False)
        os.chdir(root)
    else:
        root = Path.cwd().resolve()
        if not (root / "src/config.py").exists():
            raise FileNotFoundError("Run this notebook from the task3_agentic folder so src/ is visible.")

    req = root / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root

ROOT = ensure_project()
print("project root:", ROOT)
print("runtime:", "colab" if IN_COLAB else "local")

project root: /content/Agentic_financial_Analyser
runtime: colab


In [7]:
from src.config import describe_env, load_settings, missing_key_help

settings = load_settings()
print(describe_env(settings))
if not settings.llm_ready:
    print(missing_key_help(settings))

{'runtime': 'colab', 'secret_source': 'colab-secrets', 'ticker': 'NVDA', 'lookback': '1y', 'llm_provider': 'groq+openrouter', 'llm_model': 'openai/gpt-oss-120b:free', 'agent_model': 'openai/gpt-oss-120b:free', 'openrouter_model': 'openai/gpt-oss-120b:free', 'openrouter_key_present': 'yes', 'max_tokens': '2048', 'llm_key_present': 'yes', 'groq_key_shape': 'gsk_*', 'groq_secret_status': 'present', 'openrouter_secret_status': 'present'}


## Environment and query

The next cell loads API settings, then resolves your question to a ticker. Set `PICK` to a preset key or replace `QUERY` with your own text. The preview lists each preset with its inferred **task mode** (which tools the research agent is expected to need).

In [8]:
# Pick a preset or paste your own question below.
from src.example_questions import (
    FOLLOWUP_QUESTIONS,
    PRIMARY_QUESTIONS,
    DEFAULT_FOLLOWUP_A,
    DEFAULT_FOLLOWUP_B,
    DEFAULT_PRIMARY,
    get_followup,
    get_primary,
)
from src.task_profile import infer_task_profile
from src.ticker import parse_research_query

PICK = DEFAULT_PRIMARY  # e.g. "news_only", "price_only", "messy_summary", "small_cap_news"
QUERY = get_primary(PICK)

print("Available primary questions:", ", ".join(PRIMARY_QUESTIONS))
print()
for name, text in PRIMARY_QUESTIONS.items():
    mode = infer_task_profile(text)["mode"]
    print(f"  {name:22} mode={mode:14}  {text[:60]}{'…' if len(text) > 60 else ''}")

parsed = parse_research_query(QUERY)
profile = infer_task_profile(QUERY)
COMPANY = parsed["ticker"]
FOLLOWUP_A = get_followup(DEFAULT_FOLLOWUP_A)
FOLLOWUP_B = get_followup(DEFAULT_FOLLOWUP_B)

print("\n--- active query ---")
print("pick:", PICK)
print("question:", parsed["task"])
print("task_mode:", profile["mode"])
print("required tools:", profile["required"])
print(
    f"{parsed['subject']!r} → {parsed['ticker']}  "
    f"({parsed['name']}, via {parsed['resolved_via']}, extract {parsed['extracted_via']})"
)
print("research follow-up:", FOLLOWUP_A)
print("full-pipeline follow-up:", FOLLOWUP_B)
print("all follow-ups:", list(FOLLOWUP_QUESTIONS))

Available primary questions: full_research_apple, full_research_nvidia, news_only, news_sentiment, price_only, volatility_90d, volatility_30d, messy_summary, messy_search, small_cap_news, bare_name, bare_ticker, adaptive_open, switch_issuer

  full_research_apple    mode=full_research   Analyse the current financial health and market sentiment of…
  full_research_nvidia   mode=full_research   Analyse the current financial health and market sentiment of…
  news_only              mode=news            what is the news for apple
  news_sentiment         mode=news            what is the news sentiment for tesla
  price_only             mode=price           what is the latest apple share price
  volatility_90d         mode=volatility      what is the 90 day volatility of microsoft
  volatility_30d         mode=volatility      what is the 30 day volatility of NVDA
  messy_summary          mode=adaptive        what is the financial summary of tesla
  messy_search           mode=adaptive       

## Tool layer (sanity check)

Five LangChain tools wrap the data modules: price/technicals, realized volatility, headlines, LLM sentiment, and web search. This cell invokes two tools directly on the resolved ticker to confirm the data layer before the agents run.

In [9]:
from src.tools import ALL_TOOLS, get_price_data, calculate_volatility

print("tools:", [t.name for t in ALL_TOOLS])
print(get_price_data.invoke({"ticker": COMPANY})[:500])
print(calculate_volatility.invoke({"ticker": COMPANY, "window_days": 30}))

tools: ['get_price_data', 'calculate_volatility', 'get_news', 'llm_sentiment', 'web_search']
{"query": "AAPL", "ticker": "AAPL", "date": "2026-09-11", "close": 332.2699890136719, "sma_50": 317.8303741455078, "sma_200": 284.5368978118897, "sma_cross": "bullish", "rsi_14": 62.8428192638538, "macd": 3.265370598555137, "macd_signal": 1.9506483080893413, "macd_hist": 1.3147222904657956, "bb_upper": 331.47295124088066, "bb_mid": 316.62599792480466, "bb_lower": 301.77904460872867, "bb_position": "near_upper", "momentum_bias": "bullish", "bars": 251, "cached": true}
{"query": "AAPL", "ticker": "AAPL", "window_days": 30, "vol_pct": 31.4311, "method": "close-to-close annualized stdev * sqrt(252) * 100", "cached": true}


## Research agent (A)

LangGraph loop: `agent → tools → agent → … → finalize`

The model reads your question, decides which tools are needed, and produces a `DataBrief`. `ask_agent_a()` adds session memory: if the new question refers to the same issuer, prior observations are reused and only missing facts are fetched.

In [10]:
from src.agent_a import build_agent_a

agent_a = build_agent_a()
print(agent_a.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	nudge(nudge)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> finalize;
	agent -.-> nudge;
	agent -.-> tools;
	nudge --> agent;
	tools --> agent;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [11]:
from src.agent_a import format_answer
from src.memory import ask_agent_a, describe_memory, relate

print("session:", describe_memory())
print("plan:", relate(QUERY))
result = ask_agent_a(QUERY)

print("from_memory:", result.get("from_memory"))
print("task_mode:", result.get("task_mode"))
print("tool call order:")
for i, step in enumerate(result.get("tool_calls") or [], start=1):
    print(f"  {i}. {step['tool']}  {step.get('args') or step}")
if result.get("missing_after_run"):
    print("still missing:", result["missing_after_run"])
print()
print(format_answer(result))

session: {'session_id': '97cd87fdf127', 'thread_id': 'research-7f16faad', 'last_ticker': None, 'last_query': None, 'known_tickers': [], 'turns': 0, 'brief_cached': False, 'report_cached': False, 'tool_cache_files': 2, 'session_path': '/content/Agentic_financial_Analyser/logs/session.json'}
plan: {'related': False, 'ticker': 'AAPL', 'previous_query': None, 'intent': 'research', 'reuse': [], 'need_tools': [], 'reason': 'No prior question in session.'}
from_memory: False
task_mode: full_research
tool call order:
  1. get_price_data  {'ticker': 'AAPL'}
  2. calculate_volatility  {'ticker': 'AAPL', 'window_days': 90}
  3. get_news  {'ticker': 'AAPL'}
  4. calculate_volatility  {'ticker': 'AAPL', 'window_days': 30}
  5. llm_sentiment  {'ticker': 'AAPL', 'headlines': ["What a Republican 'wipeout' in the midterm elections means for investors", 'How Far Could Alphabet Stock Move On You In A Year?', 'Five Warning Signs the AI Stock Bubble Is in Its Final Stages', '‘You Have a Marriage Problem’: 

In [12]:
print("\n--- structured DataBrief ---")
if result.get("brief"):
    from pprint import pprint
    pprint(result["brief"], sort_dicts=False)
else:
    print(result.get("final_text") or "(no brief)")


--- structured DataBrief ---
{'ticker': 'AAPL',
 'company_name': 'Apple Inc.',
 'current_price': 332.27,
 'vol_30d_pct': 31.43,
 'vol_90d_pct': 29.32,
 'momentum': 'bullish',
 'rsi_14': 62.84,
 'financial_health': 'Apple Inc. shows a bullish momentum with a current price '
                     'of $332.27, supported by a 14-day RSI of 62.84, '
                     'indicating it is not overbought. The 30-day volatility '
                     'is at 31.43%, while the 90-day volatility is slightly '
                     'lower at 29.32%. The stock is near its upper Bollinger '
                     'Band, suggesting potential for a pullback. Overall, the '
                     'financial health appears stable but with caution due to '
                     'market conditions.',
 'sentiment_score': 0.02,
 'sentiment_label': 'neutral',
 'headlines': ["What a Republican 'wipeout' in the midterm elections means for "
               'investors',
               'How Far Could Alphabet Stock Mov

### Research follow-up

Ask a related question on the same issuer. The memory layer decides whether to answer from the stored brief or call gap tools (e.g. refresh price, fetch more news).

In [13]:
# Change key: refresh_price | recall_headlines | extend_news | switch_after_apple
FOLLOWUP_A = get_followup(DEFAULT_FOLLOWUP_A)

print("plan:", relate(FOLLOWUP_A))
follow_a = ask_agent_a(FOLLOWUP_A)
print("from_memory:", follow_a.get("from_memory"))
print("need_tools:", follow_a.get("need_tools"))
print("reused:", follow_a.get("reused"))
print()
print(follow_a.get("followup_answer") or format_answer(follow_a))
print()
print("session:", describe_memory())

plan: {'related': True, 'ticker': 'AAPL', 'previous_query': 'Analyse the current financial health and market sentiment of apple. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.', 'intent': 'refresh', 'reuse': ['price', 'vol_30', 'vol_90', 'news', 'sentiment', 'risks', 'hedge'], 'need_tools': ['get_price_data'], 'reason': 'Same issuer; reuse the brief and refresh the requested market facts.'}
from_memory: True
need_tools: ['get_price_data']
reused: ['price', 'vol_30', 'vol_90', 'news', 'sentiment', 'risks', 'hedge']

The latest price for Apple Inc. (AAPL) is **$332.27**. This price was reused from the previous analysis, where I noted that the stock is currently showing bullish momentum with a 14-day RSI of 62.84, indicating it is not overbought. 

Additionally, the following data points were reused from the previous analysis:
- **30-day volatility**: 31.43%
- **90-day volatility**: 29.32%
- **Sentiment score**: 0.02 (neut

## Critic agent (B) and final report

Typed handoff: `DataBrief` → `CritiqueDecision` → optional tool fulfill → revised brief → `FinalReport`.

Agent B checks for missing 90-day volatility, thin headlines, generic risks, or a hedge that ignores vol term structure. It may request **one** additional fact, then publish. The run below reuses the research agent output above to avoid duplicate fetches.

In [14]:
from src.agent_b import build_two_agent_graph

two_agent = build_two_agent_graph()
print(two_agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	researcher(researcher)
	critic(critic)
	fulfill(fulfill)
	revise(revise)
	publish(publish)
	__end__([<p>__end__</p>]):::last
	__start__ --> researcher;
	critic -.-> fulfill;
	critic -.-> publish;
	fulfill --> revise;
	researcher --> critic;
	revise --> critic;
	publish --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [15]:
from src.agent_b import format_final_report, run_two_agents

workflow = run_two_agents(QUERY, agent_a_result=result)

print("task_mode:", workflow.get("agent_a", {}).get("task_mode") or result.get("task_mode"))
print("revisions:", workflow["revisions"])
print("critique trail:")
for i, item in enumerate(workflow["critiques"], start=1):
    kind = item.get("request_kind") or ("publish" if not item.get("need_more_data") else "request")
    print(f"  {i}. {kind}: {item.get('reason')}")
if workflow.get("extra_facts"):
    print("facts B requested:", list(workflow["extra_facts"]))

task_mode: full_research
revisions: 2
critique trail:
  1. web_search: the hedge strategy does not specify how it relates to 90-day volatility
  2. web_search: the current brief lacks specific risks that could impact the share price
  3. publish: the current brief lacks specific risks that could impact the share price
facts B requested: ['web_search']


In [16]:
print(format_final_report(workflow))
print("\n--- structured FinalReport ---")
from pprint import pprint
if workflow.get("report"):
    pprint(workflow["report"], sort_dicts=False)
else:
    print("(no report)")

AAPL — final report (Agent A + Agent B)
Price 332.27  |  30d vol 31.43%  |  90d vol 29.32%  |  used_90d_vol=True
Critique rounds: 2

Financial health:
Technicals only; no fundamental statements.

Market sentiment: neutral (0.02)

Top three 90-day risks:
  (none)

Data-driven hedge:
(missing)

Critique trail:
  1. web_search: the hedge strategy does not specify how it relates to 90-day volatility
  2. web_search: the current brief lacks specific risks that could impact the share price
  3. publish: the current brief lacks specific risks that could impact the share price

--- structured FinalReport ---
{'ticker': 'AAPL',
 'financial_health': 'Technicals only; no fundamental statements.',
 'market_sentiment': 'neutral (0.02)',
 'top_risks': [],
 'hedge_or_strategy': '',
 'used_90d_vol': True,
 'critique_rounds': 2,
 'critic_notes': 'web_search: the hedge strategy does not specify how it '
                 'relates to 90-day volatility | web_search: the current brief '
                 'la

### Full-pipeline follow-up

`ask()` uses both the stored brief and the published report. On the same issuer it answers from memory when possible; otherwise it fetches only the facts required by the new question.

In [17]:
from src.memory import ask
from src.tools import get_price_data

print("session before follow-up:", describe_memory())
print("cached price call:")
print(get_price_data.invoke({"ticker": COMPANY})[:400])

# Change key: recall_hedge | recall_risks | refresh_price | extend_search
FOLLOWUP_B = get_followup(DEFAULT_FOLLOWUP_B)
print("plan:", relate(FOLLOWUP_B))
follow_b = ask(FOLLOWUP_B)
print("from_memory:", follow_b.get("from_memory"))
print("reused:", follow_b.get("reused"))
print("need_tools:", follow_b.get("need_tools"))
print()
print(follow_b.get("followup_answer") or follow_b.get("report"))
print()
print("session after follow-up:", describe_memory())

session before follow-up: {'session_id': '4439aeabde2f', 'thread_id': 'research-808f2c79', 'last_ticker': 'AAPL', 'last_query': 'Analyse the current financial health and market sentiment of apple. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.', 'known_tickers': ['AAPL'], 'turns': 3, 'brief_cached': True, 'report_cached': True, 'tool_cache_files': 8, 'session_path': '/content/Agentic_financial_Analyser/logs/session.json'}
cached price call:
{"query": "AAPL", "ticker": "AAPL", "date": "2026-09-11", "close": 332.2699890136719, "sma_50": 317.8303741455078, "sma_200": 284.5368978118897, "sma_cross": "bullish", "rsi_14": 62.8428192638538, "macd": 3.265370598555137, "macd_signal": 1.9506483080893413, "macd_hist": 1.3147222904657956, "bb_upper": 331.47295124088066, "bb_mid": 316.62599792480466, "bb_lower": 301.77904460872867, "bb_position":
plan: {'related': True, 'ticker': 'AAPL', 'previous_query': 'Analyse the current financ

## Observability

Every tool invocation is appended to `logs/agent_trace.jsonl` with timestamp, inputs, truncated output, duration, and whether the response came from cache.

In [18]:
import pandas as pd
from src.tracing import read_trace

trace = read_trace(limit=20)
frame = pd.DataFrame(trace)
cols = [c for c in ["ts", "tool", "ok", "cached", "duration_ms", "inputs", "output"] if c in frame.columns]
display(frame[cols] if cols else frame)

,ts,tool,ok,cached,duration_ms,inputs,output
0,2026-09-11T20:22:34.742996+00:00,get_price_data,True,False,782.86,"{'ticker': 'AAPL', 'period': '1y'}","{""query"": ""AAPL"", ""ticker"": ""AAPL"", ""date"": ""2..."
1,2026-09-11T20:22:35.527767+00:00,calculate_volatility,True,False,47.44,"{'ticker': 'AAPL', 'window_days': 30, 'period'...","{""query"": ""AAPL"", ""ticker"": ""AAPL"", ""window_da..."
2,2026-09-11T20:26:45.959964+00:00,get_price_data,True,True,1.01,"{'ticker': 'AAPL', 'period': '1y'}","{""query"": ""AAPL"", ""ticker"": ""AAPL"", ""date"": ""2..."
3,2026-09-11T20:26:45.961899+00:00,calculate_volatility,True,True,0.73,"{'ticker': 'AAPL', 'window_days': 30, 'period'...","{""query"": ""AAPL"", ""ticker"": ""AAPL"", ""window_da..."
4,2026-09-11T20:27:25.329001+00:00,get_price_data,True,True,5.47,"{'ticker': 'AAPL', 'period': '1y'}","{""query"": ""AAPL"", ""ticker"": ""AAPL"", ""date"": ""2..."
5,2026-09-11T20:27:25.333637+00:00,calculate_volatility,True,True,3.35,"{'ticker': 'AAPL', 'window_days': 30, 'period'...","{""query"": ""AAPL"", ""ticker"": ""AAPL"", ""window_da..."
6,2026-09-11T20:27:25.329888+00:00,calculate_volatility,True,False,17.32,"{'ticker': 'AAPL', 'window_days': 90, 'period'...","{""query"": ""AAPL"", ""ticker"": ""AAPL"", ""window_da..."
7,2026-09-11T20:27:25.330991+00:00,get_news,True,False,242.01,"{'ticker': 'AAPL', 'min_items': 8}","{""query"": ""AAPL"", ""ticker"": ""AAPL"", ""count"": 8..."
8,2026-09-11T20:27:27.394597+00:00,llm_sentiment,True,False,10627.01,"{'ticker': 'AAPL', 'headlines': ['What a Repub...","{""sentiment_score"": 0.02, ""label"": ""neutral"", ..."
9,2026-09-11T20:27:39.350208+00:00,web_search,True,False,1622.81,"{'query': 'Apple Inc. AAPL risks 90 days', 'ma...","{""query"": ""Apple Inc. AAPL risks 90 days"", ""so..."
